# Business Logic Developer
## Role: Tong

Input: `data/logistics_after_menghong.csv`
Output: `data/logistics_after_tong.csv` (handed to Vireak)

**Target decision (confirmed):** Multiclass risk classification on shipment `status`.
- Class 0 = Safe (`Delivered`)
- Class 1 = Pending (`In Transit`)
- Class 2 = Risky (`Lost`, `Returned`)
- `Unknown` (corrupted `'??'` placeholder) is dropped — not a valid business outcome.

**Downstream impact — communicate to model owners:**
`logistic_regression.ipynb` requires `multi_class='multinomial'`.
`decission_tree.ipynb` requires multiclass evaluation (3-row classification_report, not binary ROC-AUC).

## Step 1 — Load Menghong's Output, Verify Handoff

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/logistics_after_menghong.csv')
df_original = df.copy()

print(f"Shape: {df.shape}")

Shape: (139499, 29)


In [3]:
# Verify Menghong's handoff before touching anything
date_cols = ['ship_date', 'delivery_date', 'created_at', 'last_update']
for col in date_cols:
    assert df[col].isnull().sum() == 0, f"Menghong's handoff incomplete: {col} has nulls"
assert (df['delivery_days'] < 0).sum() == 0, "Menghong's handoff incomplete: negative delivery_days remain"
print("Handoff verified.")

Handoff verified.


## Step 0 (Prerequisite) — Categorical Normalization

No prior role (Naly/David/Menghong) owns text-variant cleanup on categorical columns. This is not business logic — it is a data quality gap that falls to Tong by default because no one else claimed it. It must happen before any business rule is meaningful, since rules built on dirty categories (e.g. `'BKK'` vs `'Bangkok'` treated as different cities) would silently produce wrong groupings.

### Normalize `shipping_mode` (9 raw variants → 4 real modes)

In [4]:
print("Before:", sorted(df['shipping_mode'].dropna().unique()))

mode_map = {
    'air': 'Air', 'AIR': 'Air', 'Air Freight': 'Air',
    'sea': 'Sea', 'Sea': 'Sea',
    'trk': 'Truck', 'TRK': 'Truck', 'truck': 'Truck',
    'rail': 'Rail'
}
df['shipping_mode'] = df['shipping_mode'].map(mode_map).fillna(df['shipping_mode'])

print("After:")
print(df['shipping_mode'].value_counts())

Before: ['AIR', 'Air Freight', 'Sea', 'TRK', 'air', 'rail', 'sea', 'trk', 'truck']
After:
shipping_mode
Air      44202
Truck    44112
Sea      29480
Rail     14605
Name: count, dtype: int64


### Normalize `status` (10 raw variants → 5 real categories)

`'??'` is an explicit unknown placeholder, not a case-variant typo — it is mapped to `NaN`, not folded into any real category.

In [5]:
print("Before:", sorted(df['status'].dropna().unique()))

status_map = {
    'in transit': 'In Transit', 'INTRANSIT': 'In Transit',
    'lost': 'Lost', 'LOST': 'Lost',
    'delivered': 'Delivered', 'DELIVERED': 'Delivered', 'Delivered': 'Delivered',
    'returned': 'Returned', 'RETURNED': 'Returned',
    '??': np.nan
}
df['status'] = df['status'].map(status_map).fillna(df['status'])

print("After (including the now-explicit NaN from '??'):")
print(df['status'].value_counts(dropna=False))

Before: ['??', 'DELIVERED', 'Delivered', 'INTRANSIT', 'LOST', 'RETURNED', 'delivered', 'in transit', 'lost', 'returned']
After (including the now-explicit NaN from '??'):
status
Delivered     40643
In Transit    27318
Lost          27155
Returned      26856
??            13459
NaN            4068
Name: count, dtype: int64


### Normalize `origin_city` / `destination_city` (8 raw variants → 3 real cities)

In [6]:
city_map = {
    'BKK': 'Bangkok', 'Bangkok': 'Bangkok',
    'PP': 'Phnom Penh', 'Phnompenh': 'Phnom Penh', 'Phnom Penh': 'Phnom Penh',
    'Saigon': 'Ho Chi Minh', 'HCMC': 'Ho Chi Minh', 'Ho Chi Minh': 'Ho Chi Minh'
}
df['origin_city'] = df['origin_city'].map(city_map).fillna(df['origin_city'])
df['destination_city'] = df['destination_city'].map(city_map).fillna(df['destination_city'])

print(df['origin_city'].value_counts())
print(df['destination_city'].value_counts())

origin_city
Phnom Penh     51053
Ho Chi Minh    50665
Bangkok        33693
Name: count, dtype: int64
destination_city
Phnom Penh     50858
Ho Chi Minh    50783
Bangkok        33644
Name: count, dtype: int64


### Normalize `origin_country` / `destination_country` (8 raw variants → 3 real countries)

In [7]:
country_map = {
    'VN': 'Vietnam', 'VNM': 'Vietnam', 'Vietnam': 'Vietnam',
    'KH': 'Cambodia', 'KHM': 'Cambodia', 'Cambodia': 'Cambodia',
    'TH': 'Thailand', 'Thailand': 'Thailand'
}
df['origin_country'] = df['origin_country'].map(country_map).fillna(df['origin_country'])
df['destination_country'] = df['destination_country'].map(country_map).fillna(df['destination_country'])

print(df['origin_country'].value_counts())
print(df['destination_country'].value_counts())

origin_country
Cambodia    44204
Vietnam     43951
Thailand    29526
Name: count, dtype: int64
destination_country
Vietnam     44226
Cambodia    44164
Thailand    29430
Name: count, dtype: int64


### Normalize `currency` and `payment_terms`

In [8]:
df['currency'] = df['currency'].str.upper()
print(df['currency'].value_counts())

payment_map = {
    'COD': 'COD', 'cod': 'COD',
    'prepaid': 'Prepaid', 'PrePaid': 'Prepaid',
    'collect': 'Collect',
    'credit 30d': 'Credit 30d'
}
df['payment_terms'] = df['payment_terms'].map(payment_map).fillna(df['payment_terms'])
print(df['payment_terms'].value_counts())

currency
USD    44325
VND    22232
KHR    22136
THB    22017
Name: count, dtype: int64
payment_terms
COD           36771
Prepaid       36486
Collect       18392
Credit 30d    18110
Name: count, dtype: int64


### Validation Checkpoint — Confirm Cardinality Collapsed Correctly

In [9]:
check_cols = ['shipping_mode','status','origin_city','destination_city',
              'origin_country','destination_country','currency','payment_terms']
for col in check_cols:
    print(f"{col}: {df[col].nunique()} unique — {sorted(df[col].dropna().unique())}")

shipping_mode: 4 unique — ['Air', 'Rail', 'Sea', 'Truck']
status: 5 unique — ['??', 'Delivered', 'In Transit', 'Lost', 'Returned']
origin_city: 3 unique — ['Bangkok', 'Ho Chi Minh', 'Phnom Penh']
destination_city: 3 unique — ['Bangkok', 'Ho Chi Minh', 'Phnom Penh']
origin_country: 3 unique — ['Cambodia', 'Thailand', 'Vietnam']
destination_country: 3 unique — ['Cambodia', 'Thailand', 'Vietnam']
currency: 4 unique — ['KHR', 'THB', 'USD', 'VND']
payment_terms: 4 unique — ['COD', 'Collect', 'Credit 30d', 'Prepaid']


## Step 2 — Drop Rows With Unresolved Status (`Unknown` / `??`)

These are corrupted records, not a business outcome. Including them in the target teaches the model to predict data corruption, not risk.

In [10]:
unresolved = df['status'].isnull()
print(f"Rows with unresolved/corrupted status ('??' originally): {unresolved.sum()}")

df = df[~unresolved].copy()
print(f"Shape after dropping: {df.shape}")

Rows with unresolved/corrupted status ('??' originally): 4068
Shape after dropping: (135431, 29)


## Step 3 — Build the Multiclass Risk Target

Confirmed mapping:
- `Delivered` → 0 (Safe)
- `In Transit` → 1 (Pending)
- `Lost`, `Returned` → 2 (Risky)

Note: `Lost` (cargo never arrives) and `Returned` (cargo arrives but rejected) are distinct real-world events collapsed into one class here. If finer-grained risk distinction is needed later, this collapse is not reversible without the original `status` column — which is preserved below specifically for that reason.

In [11]:
risk_map = {
    'Delivered': 0,
    'In Transit': 1,
    'Lost': 2,
    'Returned': 2
}
df['risk_target'] = df['status'].map(risk_map)

assert df['risk_target'].isnull().sum() == 0, "Unmapped status value slipped through"

print(df['risk_target'].value_counts().sort_index())
print(df['risk_target'].value_counts(normalize=True).sort_index().round(3))

AssertionError: Unmapped status value slipped through

## Step 4 — Business Logic Features

Each feature below is a hypothesis: does this signal correlate with the risk target? Designed specifically for the confirmed multiclass risk target, not generic exploration.

### 4a — Route Risk: International vs Domestic

In [12]:
# A shipment crossing a border has more handoff points (customs, carrier transfer)
# than a domestic shipment — more points of failure for loss/return.
df['is_international'] = (df['origin_country'] != df['destination_country']).astype(int)

print(df.groupby('is_international')['risk_target'].value_counts(normalize=True).unstack().round(3))

risk_target         0.0    1.0    2.0
is_international                     
0                 0.335  0.223  0.442
1                 0.333  0.224  0.443


### 4b — Shipping Mode Risk Profile

In [13]:
# Different modes carry different inherent risk: sea freight has longer transit
# (more time for loss), air freight is faster but costlier, truck/rail are regional.
mode_risk = df.groupby('shipping_mode')['risk_target'].value_counts(normalize=True).unstack().round(3)
print(mode_risk)

risk_target      0.0    1.0    2.0
shipping_mode                     
Air            0.333  0.223  0.444
Rail           0.334  0.223  0.443
Sea            0.337  0.221  0.442
Truck          0.331  0.227  0.442


### 4c — Delivery Duration Bucket

Uses `delivery_days` from Menghong's handoff.

In [14]:
# Unusually long transit time may correlate with risk — but this must be checked
# only on resolved shipments (Delivered/Lost/Returned), since In Transit shipments
# have a delivery_days value computed against today/last_update, not a true delivery.
df['delivery_duration_bucket'] = pd.cut(
    df['delivery_days'],
    bins=[-1, 3, 7, 14, 30, float('inf')],
    labels=['0-3d', '4-7d', '8-14d', '15-30d', '30d+']
)

print(df.groupby('delivery_duration_bucket')['risk_target'].value_counts(normalize=True).unstack().round(3))

risk_target                 0.0    1.0    2.0
delivery_duration_bucket                     
0-3d                      0.342  0.255  0.403
4-7d                      0.334  0.208  0.458
8-14d                     0.340  0.227  0.433
15-30d                    0.325  0.226  0.449
30d+                      0.333  0.224  0.443


### 4d — Payment Terms Risk Signal

In [15]:
# Hypothesis: COD (cash on delivery) shipments may have higher return rates
# than prepaid, since the customer can still refuse at the door.
payment_risk = df.groupby('payment_terms')['risk_target'].value_counts(normalize=True).unstack().round(3)
print(payment_risk)

risk_target      0.0    1.0    2.0
payment_terms                     
COD            0.334  0.225  0.441
Collect        0.336  0.225  0.439
Credit 30d     0.333  0.223  0.444
Prepaid        0.332  0.220  0.448


### 4e — High-Value Shipment Flag

Uses `total_cost_usd` from Naly's handoff.

In [16]:
# Hypothesis: high-value shipments may receive more careful handling (lower risk)
# OR may be more attractive targets for loss/theft (higher risk) — direction
# is not assumed here, only the feature is created; the model determines direction.
cost_threshold = df['total_cost_usd'].quantile(0.75)
df['is_high_value'] = (df['total_cost_usd'] >= cost_threshold).astype(int)

print(f"High-value threshold (75th percentile): ${cost_threshold:.2f}")
print(df.groupby('is_high_value')['risk_target'].value_counts(normalize=True).unstack().round(3))

High-value threshold (75th percentile): $2295.53
risk_target      0.0    1.0    2.0
is_high_value                     
0              0.334  0.224  0.442
1              0.332  0.223  0.444


### 4f — Weekend Shipment Flag

Uses `ship_weekday` from Menghong's handoff.

In [17]:
# Hypothesis: weekend-shipped cargo may have reduced staffing/handling oversight.
df['shipped_on_weekend'] = (df['ship_weekday'] >= 5).astype(int)

print(df.groupby('shipped_on_weekend')['risk_target'].value_counts(normalize=True).unstack().round(3))

risk_target           0.0    1.0    2.0
shipped_on_weekend                     
0                   0.333  0.225  0.442
1                   0.335  0.222  0.444


## Step 5 — Review Feature Signal Strength Before Handoff

Not every feature built above will actually separate the classes. Inspect the cross-tabulations from Step 4 — a feature where all classes have near-identical proportions across its categories carries no signal and should be flagged to Vireak as a candidate for exclusion, not silently included.

In [18]:
# Quick variance check across groups — larger spread = more separating power
for col in ['is_international', 'shipping_mode', 'delivery_duration_bucket',
            'payment_terms', 'is_high_value', 'shipped_on_weekend']:
    grouped = df.groupby(col)['risk_target'].value_counts(normalize=True).unstack()
    spread = grouped.std().mean()
    print(f"{col}: avg std across risk classes = {spread:.4f}")

is_international: avg std across risk classes = 0.0009
shipping_mode: avg std across risk classes = 0.0021
delivery_duration_bucket: avg std across risk classes = 0.0151
payment_terms: avg std across risk classes = 0.0028
is_high_value: avg std across risk classes = 0.0010
shipped_on_weekend: avg std across risk classes = 0.0015


## Step 6 — Final Validation

In [19]:
print("=== FINAL BUSINESS LOGIC AUDIT ===")
print(f"risk_target nulls: {df['risk_target'].isnull().sum()}")
print(f"risk_target classes: {sorted(df['risk_target'].unique())}")
print(f"\nOriginal shape: {df_original.shape}")
print(f"Final shape:    {df.shape}")
print(f"Rows removed:   {df_original.shape[0] - df.shape[0]}")

=== FINAL BUSINESS LOGIC AUDIT ===
risk_target nulls: 13459
risk_target classes: [np.float64(0.0), np.float64(nan), np.float64(1.0), np.float64(2.0)]

Original shape: (139499, 29)
Final shape:    (135431, 34)
Rows removed:   4068


## Step 7 — Save and Hand Off to Vireak

In [20]:
df.to_csv('../data/logistics_after_tong.csv', index=False)
print("Tong's task complete. Handed off to Vireak for feature engineering.")
print("\nMUST COMMUNICATE to model owners: risk_target is MULTICLASS (3 classes),")
print("not binary. logistic_regression.ipynb needs multi_class='multinomial'.")
print("decission_tree.ipynb needs multiclass evaluation, not binary ROC-AUC.")

Tong's task complete. Handed off to Vireak for feature engineering.

MUST COMMUNICATE to model owners: risk_target is MULTICLASS (3 classes),
not binary. logistic_regression.ipynb needs multi_class='multinomial'.
decission_tree.ipynb needs multiclass evaluation, not binary ROC-AUC.
